## Importing

In [1]:
##Packages:
import pandas as pd
from sqlalchemy import create_engine

#Creating Engine:
engine = create_engine('sqlite:///my_database.db')

In [2]:
##Dataset: 
df = pd.read_csv('switch_database.csv')

##Converting CSV to SQL Table:
df.to_sql('Keyboards', con=engine, if_exists = 'replace', index = False)

173

In [3]:
df

,Section,Company,Switch,Type,Spring Weight,Price (Per Switch),Sound Profile,Factory Lubed?,Availiblity Status,Release Year
0,Y-A1,Gateron,EF Curry,Linear,60g,0.20,Thocky,Yes,Active,2024.0
1,Y-A2,Gazzew,Boba U4T,Tactile,65g,0.65,Very Thocky,No,Active,2021.0
2,Y-A3,Gazzew,LT Linear,Linear,55g,0.65,Thocky,Yes,Active,2021.0
3,Y-A4,Everglide,Amber Orange,Linear,45g,0.60,Thocky,No,Active,2021.0
4,Y-A5,Durock,T1,Tactile,67g,0.60,Thocky,Yes,Active,2021.0
...,...,...,...,...,...,...,...,...,...,...
168,Z-M4,Moyu,Snow Grape,Linear,58g,0.50,Clacky,Yes,Discontinued,2023.0
169,Z-M5,Momoka,Shark Tactile,Tactile,65g,0.63,Thocky,Yes,Active,2021.0
170,Z-M6,Gateron,Type R,Tactile,55g,0.50,Thocky,Yes,Active,2025.0
171,Z-M7,SP Star,Polaris Purple,Tactile,67g,0.58,Resonant,No,Active,2021.0


## Analysis

### **Question 1:** Which Keyboard Companies Have the Most Switches? (Atleast 5 count)

In [4]:
##Query: 
query = """
SELECT Company, COUNT(1) as Switch_Count 
FROM Keyboards 
GROUP BY Company 
HAVING COUNT(1) >= 5 
ORDER BY Switch_Count DESC;
"""
company_df = pd.read_sql(query, con=engine)
company_df.head()

,Company,Switch_Count
0,Kailh,25
1,Gateron,25
2,Outemu,17
3,TTC,14
4,Cherry,13


#### **Answer:** The companies with the most switches are: 
1) **Kahil** & **Gateron** with 25 switches
2) **Outemu** with 17 switches
3) **TTC** with 14 switches
4) **Cherry** with 13 switches

### **Question 2:** What Percentages of Switches are Factory Lubed? 

In [5]:
##Query: 
query = """
SELECT `Factory Lubed?`, ROUND(AVG(CASE WHEN `Factory Lubed?` = 'Yes' THEN 1 ELSE 0 END),2) as Lube_Percent
FROM Keyboards
ORDER BY `Factory Lubed?` DESC
"""

factory_df = pd.read_sql(query, con=engine)
factory_df.head()

,Factory Lubed?,Lube_Percent
0,Yes,0.55


#### **Answer:** About 55% of our switches are factory lubed. 

### **Question 3:** What Percentage of Switches are Linear, Tactile, Clicky, and Silent? 

In [6]:
## Query: 
query = """
SELECT CASE WHEN Type IN ('Silent Linear', 'Silent Tactile') THEN 'Silent' ELSE Type END AS Updated_Type, 
    COUNT(*) AS Count, ROUND(COUNT(*)*100 / (SELECT Count(*) FROM Keyboards),2) AS Percentage
FROM Keyboards
GROUP BY Type 
ORDER BY Count DESC
"""

type_df = pd.read_sql(query, con=engine)
type_df

,Updated_Type,Count,Percentage
0,Linear,87,50.0
1,Tactile,49,28.0
2,Clicky,23,13.0
3,Silent,9,5.0
4,Silent,5,2.0


#### **Answer:** The most popular switch type in our database are linears, making up exactly half of our population

### **Question 4:** What switches are above $1? 

In [7]:
query = """
SELECT *
FROM Keyboards
WHERE `Price (Per Switch)` > 1 AND `Price (Per Switch)` != '???' 
ORDER BY `Price (Per Switch)` DESC
"""

price_df = pd.read_sql(query, con=engine)
price_df

,Section,Company,Switch,Type,Spring Weight,Price (Per Switch),Sound Profile,Factory Lubed?,Availiblity Status,Release Year
0,Y-F7,Outemu,Kitcrazy U4T,Tactile,62g,5.00,Thocky,Yes,Limited,2023.0
1,Y-E1,Bolsa Supply,Corsa,Tactile,60g,1.50,Thocky,No,Discontinued,2022.0
2,Y-F3,Huano,Strawberry Jelly V1,Tactile,45g,1.50,Clacky,Yes,Discontinued,2022.0
3,Y-B3,Gateron,Dreaming Girl Usagi,Linear,55g,1.25,Creamy,Yes,Limited,2021.0
4,Y-E2,Bolsa Supply,Techno Violet,Linear,58g,1.25,Thocky,Yes,Discontinued,2022.0
5,Z-I4,Zeal,Zealios V2,Tactile,67g,1.20,Thocky,No,Active,2018.0
6,Z-M1,Zeal,Tealios V2,Linear,67g,1.20,Creamy,No,Limited,2018.0
7,Y-D1,Moyu,Black,Tactile,67g,1.00,Clacky,Yes,Active,2020.0
8,Y-E3,Bolsa Supply,Zaku,Linear,64g,1.00,Clacky,Yes,Discontinued,2021.0
9,Z-I6,Zeal,Zealios V1,Tactile,67g,1.00,Clacky,No,Discontinued,2016.0


#### **Answer:** The most expensive switch in our list is the Kitcrazy U4T, with a selling price of $5 per switch. The rest of the switches in our list are roughly closer to a dollar each.

### **Question 5:** What are the heaviest/lightest switches?

In [9]:
query = """
SELECT *
FROM Keyboards
ORDER BY `Spring Weight` ASC
LIMIT 5
"""

light_df = pd.read_sql(query, con=engine)
light_df

,Section,Company,Switch,Type,Spring Weight,Price (Per Switch),Sound Profile,Factory Lubed?,Availiblity Status,Release Year
0,Y-B2,Varmilo,Parameter,Linear,35g,???,Clacky,No,Discontinued,2020.0
1,Y-B4,Everglide,Sakura Pink,Linear,35g,0.65,Creamy,No,Active,2020.0
2,Y-H5,Outemu,Peach V2,Silent Linear,35g,0.30,Muted,Yes,Active,2023.0
3,Z-F4,Gateron,KS-8 White,Linear,35g,0.18,Very Thocky,No,Limited,2020.0
4,Z-K1,Kailh,Silent Pink,Silent Linear,35g,0.55,Muted,No,Active,2020.0


In [10]:
query = """
SELECT *
FROM Keyboards
ORDER BY `Spring Weight` DESC
LIMIT 5
"""

heavy_df = pd.read_sql(query, con=engine)
heavy_df

,Section,Company,Switch,Type,Spring Weight,Price (Per Switch),Sound Profile,Factory Lubed?,Availiblity Status,Release Year
0,Z-K8,Kailh,Box Ancient Grey,Linear,95g,0.43,Thocky,No,Active,2018.0
1,Y-C6,Kahil,Blueberry,Tactile,80g,0.65,Thocky,No,Limited,2020.0
2,Z-A4,Cherry,Green,Clicky,80g,0.50,Resonant,No,Discontinued,1989.0
3,Z-B1,Cherry,Grey,Tactile,80g,0.50,Thocky,No,Discontinued,1989.0
4,Z-H6,KeBo,Arctos,Linear,72g,0.30,Thocky,Yes,Active,2019.0


#### **Answer:** While all of the lightest switches share the spring weight of 35g, our heaviest switch is the Box Ancient Grey, with a weight of 95g. This is a 15g difference from the 2nd heaviest, resulting in a fairly large gap between the reamining heavy switches.